# XAI Adapter Quickstart

A short walkthrough for instantiating XAI methods through `src.xai_adapter` and rendering the results as tables.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from src.data_loaders import UnifiedDataLoader, XAIDatasetParser
from src.xai_adapter import create_coxam_xai_method, create_xai_method

assets_root = repo_root / "assets"

## Feature Attribution Method

`lofo` works without SHAP, LIME, Captum, or torch, so it is a useful smoke-test for the sklearn-like `.fit()` / `.explain()` API.

In [ ]:
def predict_proba(X):
    X = np.asarray(X, dtype=float)
    p1 = np.clip(0.2 + 0.35 * X[:, 0] + 0.45 * X[:, 1] - 0.10 * X[:, 2], 0.0, 1.0)
    return np.column_stack([1.0 - p1, p1])

X_train = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
])
X_test = np.array([[1.0, 0.5, 0.0]])

lofo = create_xai_method("lofo", predict_fn=predict_proba).fit(X_train)
lofo_result = lofo.explain(X_test)

pd.DataFrame(lofo_result.values, columns=["v0", "v1", "v2"])

## CSV-Backed Explanations

Use the CSV adapter when instances, AI predictions, and explanation vectors already come from an external pipeline.

In [ ]:
csv_df = pd.DataFrame([
    {"instanceId": 0, "pred": 1, "v0": 0.2, "v1": 0.7, "v2": 0.1, "a0_i": 0.4, "a1_i": 0.5, "a2_i": 0.1},
    {"instanceId": 1, "pred": 0, "v0": 0.8, "v1": 0.1, "v2": 0.3, "a0_i": 0.6, "a1_i": 0.2, "a2_i": 0.2},
])

csv_dataset = XAIDatasetParser.from_dataframe(csv_df)
csv_method = create_xai_method("csv", dataset=csv_dataset)
records = csv_dataset.get_records([0, 1])

pd.DataFrame([
    {
        "instanceId": record.instance_id,
        "ai_prediction": record.ai_prediction,
        "features": record.features.tolist(),
        "explanation": record.explanation.tolist(),
    }
    for record in records
])

## Generate Surrogates From A New Dataset

When a CSV has instances and AI predictions but no precomputed CoXAM tables, fit `rules` and `weights` directly from `X, y`.

In [ ]:
surrogate_df = pd.DataFrame([
    {"instanceId": 0, "pred": 0, "v0": 0.0, "v1": 0.0, "v2": 0.0},
    {"instanceId": 1, "pred": 0, "v0": 0.2, "v1": 0.1, "v2": 0.0},
    {"instanceId": 2, "pred": 1, "v0": 0.9, "v1": 0.8, "v2": 1.0},
    {"instanceId": 3, "pred": 1, "v0": 1.0, "v1": 0.9, "v2": 1.0},
])

X = surrogate_df[["v0", "v1", "v2"]].to_numpy()
y = surrogate_df["pred"].to_numpy()

rules = create_xai_method("rules", app_id="toy", model_name="external", depth=2).fit(X, y)
weights = create_xai_method("weights", app_id="toy", model_name="external", variant="sparse", top_k=2).fit(X, y)

display(pd.DataFrame(rules.explain(X).values).add_prefix("rule_path_v"))
display(pd.DataFrame(weights.explain(X).values).add_prefix("weight_contribution_v"))

## CoXAM Rules vs Weights

Use `create_coxam_xai_method(...)` when the explanations live in `assets/explanations/coxam` and should be loaded through `UnifiedDataLoader`.

In [ ]:
try:
    coxam_loader = UnifiedDataLoader.from_assets(source="coxam", assets_root=str(assets_root))
    instance_ids = [0, 1, 2]
    X = coxam_loader.get_features(instance_ids, normalize=False)

    rules = create_coxam_xai_method(
        coxam_loader,
        method_type="rules",
        app_id="wine_quality",
        model_name="mlp",
        depth=3,
    )
    weights = create_coxam_xai_method(
        coxam_loader,
        method_type="weights",
        app_id="wine_quality",
        model_name="mlp",
        variant="sparse",
    )

    rules_result = rules.explain(X)
    weights_result = weights.explain(X)

    display(pd.DataFrame({
        "instanceId": instance_ids,
        "rules_prediction": [item["class_index"] for item in rules_result.metadata["predictions"]],
        "weights_probability": weights_result.metadata["predictions"],
    }))
    display(pd.DataFrame(weights_result.values).add_prefix("weight_contribution_v"))
except Exception as exc:
    pd.DataFrame([{"status": "CoXAM assets example skipped", "reason": f"{type(exc).__name__}: {exc}"}])